In [ ]:
%matplotlib widget

In [ ]:
import os
import hyperspy.api as hs
import matplotlib.pyplot as plt
import numpy as np
from skimage.filters import threshold_otsu, gaussian

import pyxem as pxm
from pathlib import Path

In [ ]:
s_list = []
s_sum_list = []

data_folder = Path("data")
files = sorted(data_folder.glob("*.zspy"))

for file in files:
    s = hs.load(file, lazy=True)
    s_list.append(s)
    s_sum = s.sum(axis=(-1, 2))
    s_sum.compute()
    s_sum_list.append(s_sum)

In [ ]:
fig, ax = plt.subplots()

2.4, 5.18 med mask
`ax.imshow(s_bs_lp.isig[0])`

2.1, 4.97 uten mask
`ax.imshow(s_bs.get_linear_plane().isig[0])`

In [ ]:
s_bs_corr_list = []
for i in range(len(s_list)): # for i in range(20, 21):
    s = s_list[i]
    s_sum = s_sum_list[i]
    s_bs = s.get_direct_beam_position(method="center_of_mass")
    s_bs.compute()
    s_sum_gauss = gaussian(s_sum.data, 1.5)
    thresh = threshold_otsu(s_sum_gauss)
    mask = s_sum_gauss < thresh
    s_bs_lp = s_bs.get_linear_plane(mask=mask)
    s_bs_corr_list.append(s_bs-s_bs_lp)

In [ ]:
# pxm.utils.plotting.plot_beam_shift_color(s_bs_corr_list[1], ax=ax, ax_indicator=False, autolim_sigma=2)

In [ ]:
s_bs_rot_list = []

for s_bs_corr in s_bs_corr_list:
    angle = np.deg2rad(30)
    
    R = np.array([
        [np.cos(angle), -np.sin(angle)],
        [np.sin(angle),  np.cos(angle)],
    ])
    
    s_bs_rot = s_bs_corr.deepcopy()
    s_bs_rot.data = s_bs_corr.data @ R.T
    s_bs_rot_list.append(s_bs_rot)

In [ ]:
# ax.imshow(s_bs_rot_list[0].inav[14.:18., 14.:18.].isig[0])

In [ ]:
if not os.path.isdir("data_shift"):
    os.mkdir("data_shift")

for i, file in enumerate(files):
    s_bs_rot = s_bs_rot_list[i]
#    if "Shifts" in s_bs_rot.metadata:
#        del s_bs_rot.metadata.Shifts
    s_bs_rot.save(f"data_shift/{file.stem}.hspy")